In [18]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder
from econml.dml import LinearDML
from lightgbm import LGBMClassifier

train = pd.read_csv("../Data/Processed/train.csv")
test = pd.read_csv("../Data/Processed/test.csv")
print(train.shape, test.shape)

(51200, 13) (12800, 13)


In [19]:
TREATMENT_COL = "segment"
OUTCOME_COL = "conversion"
CONTROL = "No E-Mail"
ID_COL = "Customer ID"

FEATURE_COLS = ["recency", "history", "mens", "womens", "newbie", "history_floor", "zip_code", "channel"]

LGBM_PARAMS = dict(n_estimators=200, learning_rate=0.05, num_leaves=15, min_child_samples=100,
                   subsample=0.8, subsample_freq=1, colsample_bytree=0.8, random_state=42, verbose=-1)

cat_ohe = ["zip_code", "channel"]

In [20]:
preprocessor=ColumnTransformer([
    ('history_log',FunctionTransformer(np.log1p,feature_names_out='one-to-one'),['history']),
    ('cat_ohe',OneHotEncoder(drop=["Rural", "Multichannel"],handle_unknown='ignore',sparse_output=False),cat_ohe)
],remainder='drop',verbose_feature_names_out=False
).set_output(transform='pandas')

In [23]:
class DMLPipeline:
    
    def __init__(self, preprocessor, estimator, feature_cols, treatment_col, outcome_col, control):
        self.preprocessor = preprocessor
        self.estimator = estimator
        self.feature_cols = feature_cols
        self.treatment_col = treatment_col
        self.outcome_col = outcome_col
        self.control = control
        
    def fit(self, df):
        X = self.preprocessor.fit_transform(df[self.feature_cols])    
        self.estimator.fit(Y=df[self.outcome_col].values, T=df[self.treatment_col].values, X=X.values)
        self.arms_ = [a for a in sorted(df[self.treatment_col].unique()) if a != self.control]
        self.feature_names_ = list(X.columns)
        return self
    
    def effects(self, df):
        X = self.preprocessor.transform(df[self.feature_cols]).values
        out = {f"ite_{arm.replace(' E-Mail', '').lower()}":
               np.ravel(self.estimator.effect(X, T0=self.control, T1=arm)) for arm in self.arms_}
        return pd.DataFrame(out, index=df.index)
        
    def recommend(self, df):
        eff = self.effects(df)
        best = pd.Series(self.arms_, index=eff.columns).reindex(eff.idxmax(axis=1)).values
        best = np.where(eff.max(axis=1) > 0, best, self.control)
        return eff.assign(best_segment=best)

In [25]:
dml=LinearDML(
    model_t=LGBMClassifier(**LGBM_PARAMS),
    model_y=LGBMClassifier(**LGBM_PARAMS),
    discrete_outcome=True,
    discrete_treatment=True,
    cv=5,
    random_state=42
)

dml_pipe=DMLPipeline(preprocessor,dml,FEATURE_COLS,TREATMENT_COL,OUTCOME_COL,CONTROL)

dml_pipe.fit(train)

print("fitted. e-mail arms compared with", CONTROL, "->", dml_pipe.arms_)
print("features seen by the model:", dml_pipe.feature_names_)

fitted. e-mail arms compared with No E-Mail -> ['Mens E-Mail', 'Womens E-Mail']
features seen by the model: ['history', 'zip_code_Suburban', 'zip_code_Urban', 'channel_Phone', 'channel_Web']


In [26]:
rec = dml_pipe.recommend(test)
rec.insert(0, ID_COL, test[ID_COL].values)
display(rec.head())

print("average estimated effect on the test set (percentage points):")
print((rec[["ite_mens", "ite_womens"]].mean() * 100).round(3).to_string())
print("\nspread across customers (std, pp):")
print((rec[["ite_mens", "ite_womens"]].std() * 100).round(3).to_string())
print("\nbest option per customer:")
print(rec["best_segment"].value_counts().to_string())

,Customer ID,ite_mens,ite_womens,best_segment
0,C-43252,0.010040,0.001854,Mens E-Mail
1,C-7410,0.004524,0.004260,Mens E-Mail
2,C-6444,0.007925,0.004724,Mens E-Mail
3,C-29292,0.006412,0.004609,Mens E-Mail
4,C-9606,0.010344,0.005992,Mens E-Mail


average estimated effect on the test set (percentage points):
ite_mens      0.671
ite_womens    0.306

spread across customers (std, pp):
ite_mens      0.273
ite_womens    0.179

best option per customer:
best_segment
Mens E-Mail      11821
Womens E-Mail      979
